In [ ]:
import random
from pathlib import Path
import time
from IPython.display import clear_output
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
from google.colab import drive
from IPython.display import display, HTML

drive.mount('/content/drive')


# =========================================================
# 1. ENV
# =========================================================

class TagMARLEnv:
    """
    1 Pacman vs 2 Ghosts
    Partial observability with ray radar + MAPPO global critic state

    Pacman actions:
      0: UP
      1: DOWN
      2: LEFT
      3: RIGHT
      4: STAY
      5: PLACE_UP
      6: PLACE_DOWN
      7: PLACE_LEFT
      8: PLACE_RIGHT
    """

    ACTIONS = {
        0: (-1, 0),   # UP
        1: (1, 0),    # DOWN
        2: (0, -1),   # LEFT
        3: (0, 1),    # RIGHT
        4: (0, 0),    # STAY
    }

    PLACE_ACTION_TO_MOVE = {
        5: 0,  # PLACE_UP
        6: 1,  # PLACE_DOWN
        7: 2,  # PLACE_LEFT
        8: 3,  # PLACE_RIGHT
    }

    RAYS = [
        (-1, 0), (-1, 1), (0, 1), (1, 1),
        (1, 0), (1, -1), (0, -1), (-1, -1),
    ]

    MAP_LIBRARY = {
        "empty": [
            "###########",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "###########",
        ],
        "medium": [
            "###########",
            "#000000000#",
            "#000#00000#",
            "#000#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00000#",
            "#000000000#",
            "###########",
        ],
        "hard": [
            "###########",
            "#0000#0000#",
            "#0##0#0##0#",
            "#0000#0000#",
            "###0000#0##",
            "#000##0000#",
            "#0#000##00#",
            "#000#000#0#",
            "#0##0#0000#",
            "#000000000#",
            "###########",
        ],
    }

    def __init__(
        self,
        max_steps=400,
        seed=42,
        min_start_dist=4,
        pacman_view_range=10,
        ghost_view_range=2,
        map_name="medium",
        wall_cooldown_steps=20,
        wall_duration_steps=15,
    ):
        self.max_steps = max_steps
        self.rng = random.Random(seed)
        self.min_start_dist = min_start_dist

        self.pacman_view_range = pacman_view_range
        self.ghost_view_range = ghost_view_range

        self.wall_cooldown_steps = wall_cooldown_steps
        self.wall_duration_steps = wall_duration_steps

        if map_name not in self.MAP_LIBRARY:
            raise ValueError(f"Unknown map_name: {map_name}")

        self.raw_map = self.MAP_LIBRARY[map_name]
        self.map_name = map_name

        self.H = len(self.raw_map)
        self.W = len(self.raw_map[0])

        self.empty_cells = [
            (r, c)
            for r in range(self.H)
            for c in range(self.W)
            if self.raw_map[r][c] == "0"
        ]

        self.reset()

    def reset(self):
        self.grid = [list(row) for row in self.raw_map]

        while True:
            sampled_positions = self.rng.sample(self.empty_cells, 3)
            pacman = sampled_positions[0]
            ghosts = sampled_positions[1:]
            if all(self.manhattan(pacman, g) >= self.min_start_dist for g in ghosts):
                self.pacman = pacman
                self.ghosts = ghosts
                break

        self.steps = 0
        self.done = False

        self.temp_walls = {}
        self.wall_cooldown = 0

        return self.get_obs()

    def in_bounds(self, r, c):
        return 0 <= r < self.H and 0 <= c < self.W

    def is_static_wall(self, r, c):
        if not self.in_bounds(r, c):
            return True
        return self.grid[r][c] == "#"

    def is_temp_wall(self, r, c):
        return (r, c) in self.temp_walls

    def is_blocked(self, r, c):
        if not self.in_bounds(r, c):
            return True
        if self.is_static_wall(r, c):
            return True
        if self.is_temp_wall(r, c):
            return True
        return False

    def move(self, pos, action):
        dr, dc = self.ACTIONS[action]
        nr, nc = pos[0] + dr, pos[1] + dc
        if self.is_blocked(nr, nc):
            return pos
        return (nr, nc)

    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _update_temp_walls(self):
        expired = [pos for pos, expire_step in self.temp_walls.items() if self.steps >= expire_step]
        for pos in expired:
            del self.temp_walls[pos]

    def _tick_wall_cooldown(self):
        if self.wall_cooldown > 0:
            self.wall_cooldown -= 1

    def _can_place_wall(self):
        return self.wall_cooldown <= 0

    def _place_temp_wall(self, pos):
        if pos == self.pacman:
            return
        if pos in self.ghosts:
            return
        if self.is_static_wall(pos[0], pos[1]):
            return

        self.temp_walls[pos] = self.steps + self.wall_duration_steps
        self.wall_cooldown = self.wall_cooldown_steps

    def _resolve_pacman_action(self, pacman_action):
        old_pos = self.pacman

        if pacman_action in self.ACTIONS:
            self.pacman = self.move(self.pacman, pacman_action)
            return

        if pacman_action in self.PLACE_ACTION_TO_MOVE:
            move_action = self.PLACE_ACTION_TO_MOVE[pacman_action]
            new_pos = self.move(self.pacman, move_action)
            moved = (new_pos != old_pos)
            self.pacman = new_pos

            if moved and self._can_place_wall():
                self._place_temp_wall(old_pos)
            return

        raise ValueError(f"Unknown pacman action: {pacman_action}")

    def _ray_scan_for_ghosts(self, start_pos, view_range):
        sx, sy = start_pos
        feat = []
        ghost_set = set(self.ghosts)

        for dr, dc in self.RAYS:
            static_wall_dist = 0
            temp_wall_dist = 0

            ghost_found = False
            ghost_dist = 0
            ghost_dx = 0
            ghost_dy = 0
            ghost_mask = 0

            r, c = sx, sy
            for step in range(1, view_range + 1):
                r += dr
                c += dc

                if not self.in_bounds(r, c):
                    static_wall_dist = step
                    break

                if self.is_static_wall(r, c):
                    static_wall_dist = step
                    break

                if self.is_temp_wall(r, c):
                    temp_wall_dist = step
                    break

                if not ghost_found and (r, c) in ghost_set:
                    ghost_found = True
                    ghost_dist = step
                    ghost_dx = r - sx
                    ghost_dy = c - sy
                    ghost_mask = 1
            else:
                static_wall_dist = view_range + 1

            if static_wall_dist == 0:
                static_wall_dist = view_range + 1
            if temp_wall_dist == 0:
                temp_wall_dist = view_range + 1

            feat.extend([
                static_wall_dist / (view_range + 1),
                temp_wall_dist / (view_range + 1),
                ghost_dist / max(1, view_range),
                ghost_dx / self.H,
                ghost_dy / self.W,
                float(ghost_mask),
            ])

        wall_ready = 1.0 if self._can_place_wall() else 0.0
        cooldown_ratio = self.wall_cooldown / max(1, self.wall_cooldown_steps)
        feat.extend([wall_ready, cooldown_ratio])

        return np.array(feat, dtype=np.float32)

    def _ray_scan_ghost_dual(self, ghost_idx, view_range):
        sx, sy = self.ghosts[ghost_idx]
        ally_set = set(self.ghosts[j] for j in range(len(self.ghosts)) if j != ghost_idx)
        feat = []

        for dr, dc in self.RAYS:
            static_wall_dist = 0
            temp_wall_dist = 0

            pacman_found = False
            pacman_dist = 0
            pacman_dx = 0
            pacman_dy = 0
            pacman_mask = 0

            ally_found = False
            ally_dist = 0
            ally_dx = 0
            ally_dy = 0
            ally_mask = 0

            r, c = sx, sy
            for step in range(1, view_range + 1):
                r += dr
                c += dc

                if not self.in_bounds(r, c):
                    static_wall_dist = step
                    break

                if self.is_static_wall(r, c):
                    static_wall_dist = step
                    break

                if self.is_temp_wall(r, c):
                    temp_wall_dist = step
                    break

                if not pacman_found and (r, c) == self.pacman:
                    pacman_found = True
                    pacman_dist = step
                    pacman_dx = self.pacman[0] - sx
                    pacman_dy = self.pacman[1] - sy
                    pacman_mask = 1

                if not ally_found and (r, c) in ally_set:
                    ally_found = True
                    ally_dist = step
                    ally_dx = r - sx
                    ally_dy = c - sy
                    ally_mask = 1
            else:
                static_wall_dist = view_range + 1

            if static_wall_dist == 0:
                static_wall_dist = view_range + 1
            if temp_wall_dist == 0:
                temp_wall_dist = view_range + 1

            feat.extend([
                static_wall_dist / (view_range + 1),
                temp_wall_dist / (view_range + 1),

                pacman_dist / max(1, view_range),
                pacman_dx / self.H,
                pacman_dy / self.W,
                float(pacman_mask),

                ally_dist / max(1, view_range),
                ally_dx / self.H,
                ally_dy / self.W,
                float(ally_mask),
            ])

        return np.array(feat, dtype=np.float32)

    def ghost_can_see_pacman(self, ghost_pos):
        gx, gy = ghost_pos

        for dr, dc in self.RAYS:
            r, c = gx, gy
            for _ in range(self.ghost_view_range):
                r += dr
                c += dc

                if not self.in_bounds(r, c):
                    break
                if self.is_static_wall(r, c):
                    break
                if self.is_temp_wall(r, c):
                    break

                if (r, c) == self.pacman:
                    return True

        return False

    def pacman_obs(self):
        return self._ray_scan_for_ghosts(self.pacman, self.pacman_view_range)  # [50]

    def ghost_obs_single(self, idx):
        return self._ray_scan_ghost_dual(idx, self.ghost_view_range)  # [80]

    def global_state(self):
        px, py = self.pacman
        coords = [px / self.H, py / self.W]

        for gx, gy in self.ghosts:
            coords.extend([gx / self.H, gy / self.W])

        pac_actor = self.pacman_obs()
        ghost_actor = np.stack([self.ghost_obs_single(i) for i in range(len(self.ghosts))], axis=0)

        gstate = np.concatenate([
            np.array(coords, dtype=np.float32),
            pac_actor,
            ghost_actor.reshape(-1),
        ], axis=0)

        return gstate.astype(np.float32)

    def get_obs(self):
        pacman_actor = self.pacman_obs()
        ghost_actor = np.stack([self.ghost_obs_single(i) for i in range(len(self.ghosts))], axis=0)

        gstate = self.global_state()

        return {
            "pacman_actor": pacman_actor,
            "pacman_critic": gstate.copy(),
            "ghost_actor": ghost_actor,
            "ghost_critic": np.stack([gstate.copy() for _ in range(len(self.ghosts))], axis=0),
        }

    def step(self, pacman_action, ghost_actions):
        if self.done:
            raise RuntimeError("Episode ended. Call reset().")

        self.steps += 1

        self._update_temp_walls()
        self._tick_wall_cooldown()

        self._resolve_pacman_action(pacman_action)
        self.ghosts = [self.move(g, a) for g, a in zip(self.ghosts, ghost_actions)]

        ghost_seen_flags = np.array(
            [1.0 if self.ghost_can_see_pacman(g) else 0.0 for g in self.ghosts],
            dtype=np.float32
        )

        pacman_reward = -1.0 if ghost_seen_flags.sum() > 0 else 0.0
        ghost_rewards = ghost_seen_flags.copy()

        if self.steps >= self.max_steps:
            self.done = True

        obs = self.get_obs()
        rewards = {
            "pacman": pacman_reward,
            "ghosts": ghost_rewards
        }
        dones = {
            "pacman": float(self.done),
            "ghosts": np.array([float(self.done)] * len(self.ghosts), dtype=np.float32)
        }
        info = {
            "temp_walls": dict(self.temp_walls),
            "wall_cooldown": self.wall_cooldown,
        }
        return obs, rewards, dones, info

    def render_text(self):
        board = [row[:] for row in self.grid]

        for (wr, wc) in self.temp_walls.keys():
            if board[wr][wc] == "0":
                board[wr][wc] = "W"

        pr, pc = self.pacman
        board[pr][pc] = "P"

        for i, (gr, gc) in enumerate(self.ghosts):
            board[gr][gc] = str(i + 1)

        print("\n".join("".join(row) for row in board))
        print(
            f"map={self.map_name}, steps={self.steps}, done={self.done}, "
            f"wall_cd={self.wall_cooldown}, temp_walls={self.temp_walls}"
        )

    def render_pretty(self):
        ghost_symbols = ["👻", "👾"]

        html = """
        <style>
        .game-board {
            border-collapse: collapse;
            font-size: 24px;
            line-height: 1;
        }
        .game-board td {
            width: 34px;
            height: 34px;
            text-align: center;
            vertical-align: middle;
            padding: 0;
            font-family: Arial, sans-serif;
        }
        .wall {
            background: #111;
            border: 1px solid #333;
        }
        .empty {
            background: #2b2b2b;
            color: #ddd;
        }
        .temp {
            background: #1e73ff;
        }
        </style>
        """

        html += f"<div style='font-family:Arial; font-size:20px; margin-bottom:8px;'>[STEP {self.steps}]</div>"
        html += "<table class='game-board'>"

        for r in range(self.H):
            html += "<tr>"
            for c in range(self.W):
                pos = (r, c)

                if self.grid[r][c] == "#":
                    html += "<td class='wall'></td>"

                elif pos in self.temp_walls:
                    html += "<td class='temp'>🟦</td>"

                elif pos == self.pacman:
                    html += "<td class='empty'>🟡</td>"

                elif pos in self.ghosts:
                    idx = self.ghosts.index(pos)
                    html += f"<td class='empty'>{ghost_symbols[idx % len(ghost_symbols)]}</td>"

                else:
                    html += "<td class='empty'>·</td>"

            html += "</tr>"

        html += "</table>"
        html += f"""
        <div style='font-family:Arial; font-size:14px; margin-top:8px;'>
            map={self.map_name} | steps={self.steps} | done={self.done} |
            wall_cd={self.wall_cooldown} | temp_walls={len(self.temp_walls)}
        </div>
        """

        display(HTML(html))

    def render_html(self, step_idx, pac_action, ghost_actions, pac_reward, ghost_rewards):
        ghost_symbols = ["👻", "👾"]

        html = """
        <style>
        .game-wrap {
            background: #242424;
            color: white;
            display: inline-block;
            padding: 12px;
            border-radius: 10px;
            font-family: Arial, sans-serif;
        }
        .game-board {
            border-collapse: collapse;
            font-size: 24px;
        }
        .game-board td {
            width: 34px;
            height: 34px;
            text-align: center;
            vertical-align: middle;
        }
        .wall { background: #111; }
        .empty { background: #2b2b2b; }
        .temp { background: #1e73ff; }
        </style>
        """

        html += "<div class='game-wrap'>"
        html += f"<div>[STEP {step_idx}]</div>"
        html += "<table class='game-board'>"

        for r in range(self.H):
            html += "<tr>"
            for c in range(self.W):
                pos = (r, c)

                if self.grid[r][c] == "#":
                    html += "<td class='wall'></td>"
                elif pos in self.temp_walls:
                    html += "<td class='temp'>🟦</td>"
                elif pos == self.pacman:
                    html += "<td class='empty'>🟡</td>"
                elif pos in self.ghosts:
                    idx = self.ghosts.index(pos)
                    html += f"<td class='empty'>{ghost_symbols[idx]}</td>"
                else:
                    html += "<td class='empty'>·</td>"

            html += "</tr>"

        html += "</table>"

        html += f"""
        <div>
            pac_action={pac_action} |
            ghost_actions={ghost_actions}
        </div>
        <div>
            pac_reward={pac_reward} |
            ghost_rewards={ghost_rewards}
        </div>
        """

        html += "</div>"
        return html


# =========================================================
# 2. MODEL
# =========================================================

class RecurrentActorCritic(nn.Module):
    def __init__(self, actor_obs_dim, critic_obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.actor_encoder = nn.Sequential(
            nn.Linear(actor_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.critic_encoder = nn.Sequential(
            nn.Linear(critic_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.actor_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)
        self.critic_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)

        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def init_hidden(self, batch_size, device):
        actor_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        actor_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        return actor_h, actor_c, critic_h, critic_c

    def forward_step(self, actor_obs, critic_obs, hidden):
        actor_h, actor_c, critic_h, critic_c = hidden

        actor_x = self.actor_encoder(actor_obs).unsqueeze(0)
        critic_x = self.critic_encoder(critic_obs).unsqueeze(0)

        actor_out, (actor_h, actor_c) = self.actor_lstm(actor_x, (actor_h, actor_c))
        critic_out, (critic_h, critic_c) = self.critic_lstm(critic_x, (critic_h, critic_c))

        actor_out = actor_out.squeeze(0)
        critic_out = critic_out.squeeze(0)

        logits = self.policy_head(actor_out)
        value = self.value_head(critic_out).squeeze(-1)

        hidden = (actor_h, actor_c, critic_h, critic_c)
        return logits, value, hidden


# =========================================================
# 3. 체크포인트 로드
# =========================================================

def load_policy_for_inference(
    checkpoint_path,
    hidden_dim=128,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    pacman_actor_dim = 50
    ghost_actor_dim = 80
    critic_obs_dim = 216
    pacman_action_dim = 9
    ghost_action_dim = 5

    pacman_net = RecurrentActorCritic(
        actor_obs_dim=pacman_actor_dim,
        critic_obs_dim=critic_obs_dim,
        action_dim=pacman_action_dim,
        hidden_dim=hidden_dim
    ).to(device)

    ghost_net = RecurrentActorCritic(
        actor_obs_dim=ghost_actor_dim,
        critic_obs_dim=critic_obs_dim,
        action_dim=ghost_action_dim,
        hidden_dim=hidden_dim
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)

    pacman_net.load_state_dict(ckpt["pacman_model_state_dict"])
    ghost_net.load_state_dict(ckpt["ghost_model_state_dict"])

    pacman_net.eval()
    ghost_net.eval()

    print(f"[LOADED] checkpoint: {checkpoint_path}")
    print(f"[LOADED] update: {ckpt.get('update', 'unknown')}")

    return pacman_net, ghost_net


# =========================================================
# 4. 액션 이름 보기용
# =========================================================

PACMAN_ACTION_NAMES = {
    0: "UP",
    1: "DOWN",
    2: "LEFT",
    3: "RIGHT",
    4: "STAY",
    5: "PLACE_UP",
    6: "PLACE_DOWN",
    7: "PLACE_LEFT",
    8: "PLACE_RIGHT",
}

GHOST_ACTION_NAMES = {
    0: "UP",
    1: "DOWN",
    2: "LEFT",
    3: "RIGHT",
    4: "STAY",
}


# =========================================================
# 5. Inference / Evaluation
# =========================================================

@torch.no_grad()
def run_inference(
    checkpoint_path,
    episodes=3,
    greedy=True,
    render=True,
    render_delay=0.3,
    device=None,
    env_seed=999,
    max_steps=400,
    pacman_view_range=10,
    ghost_view_range=2,
    map_name="medium",
    wall_cooldown_steps=20,
    wall_duration_steps=15,
    hidden_dim=128,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    env = TagMARLEnv(
        max_steps=max_steps,
        seed=env_seed,
        min_start_dist=4,
        pacman_view_range=pacman_view_range,
        ghost_view_range=ghost_view_range,
        map_name=map_name,
        wall_cooldown_steps=wall_cooldown_steps,
        wall_duration_steps=wall_duration_steps,
    )

    pacman_net, ghost_net = load_policy_for_inference(
        checkpoint_path=checkpoint_path,
        hidden_dim=hidden_dim,
        device=device,
    )

    for ep in range(episodes):
        print("\n" + "=" * 80)
        print(f"[EPISODE {ep+1}]")

        obs = env.reset()
        done = False

        pac_hidden = pacman_net.init_hidden(1, device)
        ghost_hidden = ghost_net.init_hidden(len(env.ghosts), device)

        pac_total = 0.0
        ghost_total = np.zeros(len(env.ghosts), dtype=np.float32)
        step_idx = 0

        display_handle = display(HTML(""), display_id=True)

        while not done:
            step_idx += 1

            pac_actor_obs = torch.tensor(
                obs["pacman_actor"], dtype=torch.float32, device=device
            ).unsqueeze(0)

            pac_critic_obs = torch.tensor(
                obs["pacman_critic"], dtype=torch.float32, device=device
            ).unsqueeze(0)

            ghost_actor_obs = torch.tensor(
                obs["ghost_actor"], dtype=torch.float32, device=device
            )

            ghost_critic_obs = torch.tensor(
                obs["ghost_critic"], dtype=torch.float32, device=device
            )

            pac_logits, _, pac_hidden = pacman_net.forward_step(
                pac_actor_obs, pac_critic_obs, pac_hidden
            )
            ghost_logits, _, ghost_hidden = ghost_net.forward_step(
                ghost_actor_obs, ghost_critic_obs, ghost_hidden
            )

            if greedy:
                pac_action = torch.argmax(pac_logits, dim=-1).item()
                ghost_actions = torch.argmax(ghost_logits, dim=-1).cpu().numpy().tolist()
            else:
                pac_dist = Categorical(logits=pac_logits)
                pac_action = pac_dist.sample().item()

                ghost_dist = Categorical(logits=ghost_logits)
                ghost_actions = ghost_dist.sample().cpu().numpy().tolist()

            obs, rewards, dones, info = env.step(pac_action, ghost_actions)

            pac_total += rewards["pacman"]
            ghost_total += rewards["ghosts"]
            done = bool(dones["pacman"])

            if render:
                html = env.render_html(
                    step_idx,
                    PACMAN_ACTION_NAMES[pac_action],
                    [GHOST_ACTION_NAMES[a] for a in ghost_actions],
                    rewards["pacman"],
                    rewards["ghosts"]
                )

                display_handle.update(HTML(html))
                time.sleep(render_delay)


        print(f"[RESULT] PacmanReturn={pac_total:.2f}, GhostMeanReturn={ghost_total.mean():.2f}")



In [ ]:
run_inference(
    checkpoint_path="/content/drive/MyDrive/tag_mappo_checkpoints_place4/checkpoint_update_5000.pt",
    episodes=1,
    greedy=False,
    render=True,
    render_delay=0.1,   # 0.5초마다 한 step
    device=None,
    env_seed=999,
    max_steps=1000,
    pacman_view_range=10,
    ghost_view_range=2,
    map_name="medium",
    wall_cooldown_steps=20,
    wall_duration_steps=15,
    hidden_dim=128,
)